# Search rounds and arms — step by step

Since task 029 the multi-round loop is **wired into the production runner**: a
standard or deep run repeats the acquire → screen pair until the depth's round
cap (standard 2 / deep 3) or a yield collapse (`short_circuit`). This notebook
steps through those rounds **cell by cell, mirroring the runner's gate**, so
each stage's inputs and outputs stay inspectable:

```
round 1   acquire (plain fan-out)  ->  screen
gate      evaluate_deep_stop: round cap reached? yield collapsed?
round 2   acquire (reformulate / snowball / suggest / diversity)  ->  screen
gate      ...
stop      finalise_deep_stop writes the loop's stop condition
          onto the final round's coverage row
```

For the actual production path in one shot — the runner itself — use
`run_live_deep.py` in this directory instead. This notebook mirrors it.

> **Cost.** This is live: real provider APIs, real LLM screening between
> rounds. Worst case per round ≈ 2 x `record_cap_per_backend` documents,
> each screened `SCREEN_REPS` (3) times. The config cell prints the estimate
> — read it before running the round cells.

In [1]:
import os
import sys
import time
import uuid
from datetime import UTC, datetime
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd()
while not (REPO / "backend").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "backend" / "src"))
load_dotenv(REPO / ".env")

MISSING = [k for k in ("OPENALEX_API_KEY", "OVERTON_API_KEY", "OPENAI_API_KEY")
           if not os.environ.get(k)]
print("repo:", REPO)
print("database:", (os.environ.get("DATABASE_URL") or "UNSET").rsplit("/", 1)[-1])
print("missing keys:", MISSING or "none")

repo: /Users/rosie.oxbury/Documents/git_repos/policy_atlas
database: policy_atlas
missing keys: none


In [2]:
from sqlalchemy import select

from policy_atlas.core import events
from policy_atlas.core.db import get_engine
from policy_atlas.core.schema import (
    evidence_scope,
    project,
    project_source_snapshot,
    runs,
    search_coverage_record,
)
from policy_atlas.evidence_base.assess.screen import ScreenContext, screen_sources
from policy_atlas.evidence_base.assess.screening_backend import OpenAIScreeningBackend
from policy_atlas.evidence_base.sourcing import search_generation, search_live
from policy_atlas.evidence_base.sourcing.acquire import AcquireContext
from policy_atlas.evidence_base.sourcing.search_loop import (
    CONFIDENT_FLOOR,
    DEPTH_CONSTANTS,
    SHORT_CIRCUIT_RATE,
    THIN_CONFIDENT_RELEVANT,
    confident_relevant_count,
    count_existing_rounds,
    docs_screened_from_payload,
    evaluate_deep_stop,
    finalise_deep_stop,
    new_confident_relevant_for_run,
    run_search,
)

QUERY = (
    "interventions to reduce consumption of high fat, sugar, and salt (HFSS) foods"
)
DEPTH = "deep"            # "standard" for the 2-round, reformulate-only version

constants = DEPTH_CONSTANTS[DEPTH]
per_round = constants["record_cap_per_backend"] * 2
print(f"depth={DEPTH}  round_cap={constants['round_cap']}  arms={sorted(constants['arms'])}")
print(f"stops: round cap, or short_circuit when a round yields < 1 new "
      f"confident-relevant (confidence >= {CONFIDENT_FLOOR}) per "
      f"{int(1 / SHORT_CIRCUIT_RATE)} screened")
print(f"thin-evidence overlay below {THIN_CONFIDENT_RELEVANT} confident-relevant docs")
print(f"\nworst case: ~{per_round} docs/round x {constants['round_cap']} rounds "
      f"= ~{per_round * constants['round_cap']} docs screened "
      f"(~{per_round * constants['round_cap'] * 3} model calls)")

depth=deep  round_cap=3  arms=['diversity', 'reformulate', 'snowball', 'suggest']
stops: round cap, or short_circuit when a round yields < 1 new confident-relevant (confidence >= 0.7) per 50 screened
thin-evidence overlay below 8 confident-relevant docs

worst case: ~400 docs/round x 3 rounds = ~1200 docs screened (~3600 model calls)


## One project, one scope

Every round writes into the same scope — that is what makes them rounds.
`run_search` derives its round index from how many coverage rows the scope
already has, so the second acquire call is automatically round 2 and unlocks
the arms. (This is also what makes the production gate park/resume-safe: the
round number lives in the database, not in memory.)

In [3]:
engine = get_engine()

now = datetime.now(UTC)
PROJECT_ID, SCOPE_ID = uuid.uuid4(), uuid.uuid4()
with engine.begin() as conn:
    conn.execute(project.insert().values(
        project_id=PROJECT_ID, name=f"rounds-{DEPTH}-{now:%Y%m%d-%H%M%S}",
        status="active", created_at=now, updated_at=now,
    ))
    conn.execute(evidence_scope.insert().values(
        evidence_scope_id=SCOPE_ID, project_id=PROJECT_ID, intent=QUERY,
        context={"search": {"depth": DEPTH}}, created_at=now,
    ))

ACQUIRE_CONTEXT = AcquireContext(
    scope_id=SCOPE_ID, intent=QUERY, context={"search": {"depth": DEPTH}}
)
SCREEN_CONTEXT = ScreenContext(scope_id=SCOPE_ID, intent=QUERY, context={})
generation = search_generation.OpenAISearchGenerationBackend()
screening = OpenAIScreeningBackend()

ROUND_LOG = []   # one dict per round: run ids + the numbers the gate saw
print("project:", PROJECT_ID)

project: 791292e4-7f0d-454f-8c27-e393d0cbd0eb


In [4]:
def new_run(conn):
    run_id = uuid.uuid4()
    conn.execute(runs.insert().values(
        run_id=run_id, project_id=PROJECT_ID, status="running",
        started_at=datetime.now(UTC),
    ))
    return run_id


def acquire_round():
    """One acquire run; run_search picks its round index from coverage rows."""
    with engine.begin() as conn:
        run_id = new_run(conn)
        counts = run_search(
            conn,
            project_id=PROJECT_ID,
            run_id=run_id,
            context=ACQUIRE_CONTEXT,
            backends=search_live.live_search_backends(),
            generation_backend=generation,
        )
    print(f"  acquire  round={counts['search']['round_index']}  "
          f"acquired={counts['acquired']}  returned={counts['results_returned']}  "
          f"over_cap={counts['dropped_over_cap']}")
    return run_id, counts


def screen_round():
    """One stage-1 screen run over whatever is unscreened in the scope."""
    with engine.begin() as conn:
        run_id = new_run(conn)
        counts = screen_sources(
            conn,
            project_id=PROJECT_ID,
            run_id=run_id,
            context=SCREEN_CONTEXT,
            screening_backend=screening,
        )
    print(f"  screen   screened={counts.get('screened')}  "
          f"relevant={counts.get('relevant')}")
    return run_id, counts


def run_one_round():
    """Acquire + screen, recording exactly what the runner's gate will read."""
    started = time.monotonic()
    acquire_run, acquire_counts = acquire_round()
    screen_run, screen_counts = screen_round()
    with engine.connect() as conn:
        entry = {
            "round": count_existing_rounds(
                conn, project_id=PROJECT_ID, scope_id=SCOPE_ID
            ),
            "acquire_run": acquire_run,
            "screen_run": screen_run,
            "acquired": acquire_counts["acquired"],
            "docs_screened": docs_screened_from_payload(screen_counts),
            "new_confident": new_confident_relevant_for_run(
                conn, project_id=PROJECT_ID, scope_id=SCOPE_ID, run_id=screen_run
            ),
            "confident_total": confident_relevant_count(
                conn, project_id=PROJECT_ID, scope_id=SCOPE_ID
            ),
            "seconds": time.monotonic() - started,
        }
    ROUND_LOG.append(entry)
    print(f"  round {entry['round']} done in {entry['seconds']:.0f}s: "
          f"+{entry['new_confident']} confident-relevant "
          f"(total {entry['confident_total']})")
    return entry


def gate():
    """The same stop decision the production runner makes between rounds."""
    last = ROUND_LOG[-1]
    decision = evaluate_deep_stop(
        round_index=last["round"],
        new_confident_relevant=last["new_confident"],
        docs_screened_this_round=last["docs_screened"],
        round_cap=constants["round_cap"],
    )
    if not decision.stop:
        print(f"gate: continue -> round {last['round'] + 1}")
        return decision
    thin = last["confident_total"] < THIN_CONFIDENT_RELEVANT
    with engine.begin() as conn:
        final = finalise_deep_stop(
            conn,
            project_id=PROJECT_ID,
            scope_id=SCOPE_ID,
            stop_condition=decision.stop_condition,
            thin=thin,
        )
    print(f"gate: STOP ({decision.stop_condition})"
          + (f" -> overlaid as {final}" if final != decision.stop_condition else ""))
    return decision

## Round 1 — the plain fan-out

No arms yet: reformulation needs graded exemplars, which only exist after
this round's screening.

In [5]:
print("round 1")
run_one_round()
decision = gate()

round 1
2026-08-06 16:13:44 [info     ] search_generation.queries.usage cached_tokens=0 completion_tokens=116 prompt_tokens=634 total_tokens=750
2026-08-06 16:14:17 [info     ] acquire.capped                 backend=openalex cap=200 dropped=324 kept=200 project_id=791292e4-7f0d-454f-8c27-e393d0cbd0eb run_id=98a1871b-8c92-430f-8b55-eb9b59c2c9ac
2026-08-06 16:14:17 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=inraefr-ec7857d2cdb562679224c0fdfd17c19b cap=50 tag_count=72
2026-08-06 16:14:17 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=stateofnebraska-e9fb3ab18bfc3d7afec34cb8dfc292e9 cap=50 tag_count=54
2026-08-06 16:14:18 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=scottishgovernment-0d1644bab8e4a03410ad932d8321f849 cap=50 tag_count=57
2026-08-06 16:14:18 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=governmentofoman-f730c15fdc46eb4adb4145802388a5cb cap=50 tag_cou

## Rounds 2+ — arms unlocked

Run this cell repeatedly (or let the loop finish) — it runs one more round
whenever the gate said continue. At round 2 the acquire log should show the
reformulate/diversity arms (plus snowball and suggest at deep), and the
arms-fired table below itemises them.

In [6]:
while not decision.stop:
    print(f"round {ROUND_LOG[-1]['round'] + 1}")
    run_one_round()
    decision = gate()

print(f"\nloop finished after {len(ROUND_LOG)} rounds")

round 2
2026-08-06 16:16:56 [info     ] search_generation.reformulate.usage cached_tokens=0 completion_tokens=110 prompt_tokens=2551 total_tokens=2661
2026-08-06 16:17:12 [info     ] search_generation.suggest.usage cached_tokens=0 completion_tokens=280 prompt_tokens=1906 total_tokens=2186
2026-08-06 16:17:16 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=asauk-7859e2aad73310e47c048f6f8b61cd3c cap=50 tag_count=55
2026-08-06 16:17:16 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=paho-71324655d322a82d6643ce8d88a4df8a cap=50 tag_count=61
2026-08-06 16:17:16 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=paho-67fc0762396e54ff0ed3aad9d8bfcb1d cap=50 tag_count=54
2026-08-06 16:17:18 [info     ] embed.summary                  already_embedded=400 budget_exceeded=0 embedded=242 failed=0 run_id=5f9b8589-30bd-44c0-bcec-ef5cf98646c9 skipped_no_units=0 units_embedded=496
  acquire  round=2  acquired=242  retu

## Per-round yield

`cost per new confident-relevant` is documents screened divided by new
confidently-relevant found — it should climb round on round (cheap finds come
first). When it climbs past what a relevant document is worth to you, that is
the round the loop should have stopped at; `short_circuit` is the automatic
version of that judgement.

In [7]:
print("{:>6} {:>9} {:>14} {:>12} {:>19} {:>8}".format(
    "round", "acquired", "docs screened", "new c-rel", "cost per new c-rel", "seconds"))
for entry in ROUND_LOG:
    cost = (entry["docs_screened"] / entry["new_confident"]
            if entry["new_confident"] else None)
    print("{:>6} {:>9} {:>14} {:>12} {:>19} {:>8.0f}".format(
        entry["round"], entry["acquired"], entry["docs_screened"],
        entry["new_confident"], "-" if cost is None else f"{cost:.1f}",
        entry["seconds"]))

total = sum(entry["docs_screened"] for entry in ROUND_LOG)
print(f"\ntotal screened: {total}  (~{total * 3} model calls)")

 round  acquired  docs screened    new c-rel  cost per new c-rel  seconds
     1       400            400          316                 1.3      184
     2       242            242          188                 1.3      112
     3        60             60           43                 1.4       40

total screened: 702  (~2106 model calls)


## Which arms actually fired

Every executed provider call leaves a `search.executed` event with a `verb`
and `query_origin` — together they name the arm. Zero snowball/suggest rows at
deep round 2 is a finding about round 1's screening yield (no confident
OpenAlex seeds), not a bug in the arms.

In [8]:
run_to_round = {str(entry["acquire_run"]): entry["round"] for entry in ROUND_LOG}
by_round = {}
with engine.connect() as conn:
    for event in events.read(conn, PROJECT_ID):
        if event["event_type"] != "search.executed":
            continue
        round_index = run_to_round.get(str(event["run_id"]), "?")
        payload = event["payload"]
        key = (round_index, payload["verb"], payload["query_origin"])
        bucket = by_round.setdefault(key, {"calls": 0, "records": 0})
        bucket["calls"] += 1
        bucket["records"] += payload.get("result_count") or 0

print("{:>6} {:<18} {:<18} {:>7} {:>9}".format(
    "round", "verb", "query_origin", "calls", "records"))
for (round_index, verb, origin), bucket in sorted(
    by_round.items(), key=lambda kv: str(kv[0])
):
    print("{:>6} {:<18} {:<18} {:>7} {:>9}".format(
        round_index, verb, origin, bucket["calls"], bucket["records"]))

verbs = {verb for (_r, verb, _o) in by_round}
for verb, label in (("fetch_citations", "snowball forward"),
                    ("fetch_references", "snowball backward"),
                    ("lookup_dois", "suggest (DOI grounding)"),
                    ("lookup_title", "suggest (title grounding)")):
    print(f"  {label:28s} {'yes' if verb in verbs else 'NO'}")

 round verb               query_origin         calls   records
     1 search             generated                5       318
     1 search             paraphrase               2       200
     1 search             variant_rct              5       119
     1 search             variant_sr               5       122
     1 search             verbatim                 1       100
     2 fetch_citations    snowball_forward         5        16
     2 fetch_references   snowball_backward        1        24
     2 lookup_title       suggestion_title         6         6
     2 search             generated                4       130
     2 search             paraphrase               2       200
     2 search             verbatim                 1        15
     3 fetch_citations    snowball_forward         5        24
     3 fetch_references   snowball_backward        1        16
     3 lookup_dois        suggestion_doi           1         3
     3 lookup_title       suggestion_title         5   

## Corpus and coverage at the end

One coverage row per round. Earlier rounds keep their own stop condition; the
final row carries the loop-level stop — `budget_exhausted` (round cap),
`short_circuit` (yield collapse), or `re_searched_still_thin` (finished with
fewer than 8 confidently-relevant documents).

In [9]:
with engine.connect() as conn:
    corpus = conn.execute(
        select(project_source_snapshot)
        .where(project_source_snapshot.c.project_id == PROJECT_ID)
    ).fetchall()
    coverage = conn.execute(
        select(search_coverage_record)
        .where(search_coverage_record.c.project_id == PROJECT_ID)
        .order_by(search_coverage_record.c.created_at)
    ).fetchall()

print(f"documents in corpus: {len(corpus)}")
for index, row in enumerate(coverage, start=1):
    print(f"  round {index}: stop={row.stop_condition}  adequacy={row.adequacy_verdict}")
print(f"\nproject_id = {PROJECT_ID}")

documents in corpus: 702
  round 1: stop=completed  adequacy=adequate
  round 2: stop=completed  adequacy=adequate
  round 3: stop=budget_exhausted  adequacy=adequate

project_id = 791292e4-7f0d-454f-8c27-e393d0cbd0eb
